In [11]:
import numpy as np

class Network:
    def __init__(self, sizes):
        # sizes = [input_dim, hidden..., output_dim]
        self.W = [np.random.randn(sizes[i+1], sizes[i]) * np.sqrt(1/sizes[i])
                  for i in range(len(sizes)-1)]
        self.b = [np.zeros(sizes[i+1]) for i in range(len(sizes)-1)]

    def sigmoid(self, z):       return 1 / (1 + np.exp(-z))
    def sigmoid_prime(self, z): s = self.sigmoid(z); return s * (1 - s)

    def forward(self, X):                 # X: (d, m)
        self.zs, self.acts = [], [X]
        a = X
        for W, b in zip(self.W, self.b):
            z = W @ a + b
            a = self.sigmoid(z)
            self.zs.append(z); self.acts.append(a)
        return a                          # (out, m)

    def backward(self, y):                # y: (out, m)
        m = y.shape[1]
        L = len(self.W)
        dW, db = [None]*L, [None]*L

        # output layer
        delta = (self.acts[-1] - y) * self.sigmoid_prime(self.zs[-1])
        dW[-1] = delta @ self.acts[-2].T / m
        db[-1] = np.mean(delta, axis=1)

        # walk backward through hidden layers
        for l in range(L-2, -1, -1):
            delta = (self.W[l+1].T @ delta) * self.sigmoid_prime(self.zs[l])
            dW[l] = delta @ self.acts[l].T / m
            db[l] = np.mean(delta, axis=1)
        return dW, db

    def step(self, dW, db, eta):
        for l in range(len(self.W)):
            self.W[l] -= eta * dW[l]
            self.b[l] -= eta * db[l]

In [12]:
def train(model, X, Y, epochs, eta, batch_size, verbose=True):
    m = X.shape[1]
    for epoch in range(epochs):
        idx = np.random.permutation(m)        # shuffle each epoch
        X, Y = X[:, idx], Y[:, idx]
        for start in range(0, m, batch_size):
            xb = X[:, start:start+batch_size]
            yb = Y[:, start:start+batch_size]
            model.forward(xb)
            dW, db = model.backward(yb)
            model.step(dW, db, eta)
        if verbose and epoch % (epochs//10) == 0:
            loss = np.mean((model.forward(X) - Y) ** 2)
            print(f"epoch {epoch:5d}  loss {loss:.4f}")

In [13]:
# XOR: inputs as columns, targets as columns
X = np.array([[0,0,1,1],
              [0,1,0,1]])          # shape (2, 4)
Y = np.array([[0,1,1,0]])          # shape (1, 4)

net = Network([2, 4, 1])           # 2 in → 4 hidden → 1 out
train(net, X, Y, epochs=5000, eta=2.0, batch_size=4)

print(net.forward(X))              # should be ≈ [0, 1, 1, 0]

epoch     0  loss 0.2617
epoch   500  loss 0.1580
epoch  1000  loss 0.0477
epoch  1500  loss 0.0149
epoch  2000  loss 0.0107
epoch  2500  loss 0.2550
epoch  3000  loss 0.0049
epoch  3500  loss 0.0059
epoch  4000  loss 0.0036
epoch  4500  loss 0.2533
[[0.05847088 0.94902499 0.94547756 0.04523898]]


In [32]:
X[:, np.random.permutation(4)]

array([[0, 1, 0, 1],
       [0, 0, 1, 1]])